In [ ]:
!pip install -q mne numpy scipy pandas matplotlib seaborn
!pip install -q mne pyvista pyvistaqt ipywidgets seaborn pymatreader


import os, glob
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
from scipy.signal import welch, correlate



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 28.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.2.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.0/95.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.9/145.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.3 MB/s eta 0:00:00


In [ ]:
# ============================================================
# 0) COLAB + DRIVE SETUP
#    - Mount Drive
#    - Define a project root
#    - Make MNE downloads persistent (fsaverage, etc.)
# ============================================================

from google.colab import drive
import os

drive.mount("/content/drive")

# Project root folder in Drive
PROJECT_DIR = "/content/drive/MyDrive/tiw_EO_EC_organized/preprocessed_organized/cleaned_organized"
os.makedirs(PROJECT_DIR, exist_ok=True)



print("PROJECT_DIR:", PROJECT_DIR)


Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/tiw_EO_EC_organized/preprocessed_organized/cleaned_organized


In [ ]:
#location to save results
import os

RESULTS_DIR = "/content/drive/MyDrive/tiw_EO_EC_organized/acw_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Results folder:", RESULTS_DIR)

Results folder: /content/drive/MyDrive/tiw_EO_EC_organized/acw_results


In [ ]:
# ============================================================
# A) INPUTS: data folder + which files to run
# ============================================================

import glob

DATA_DIR = PROJECT_DIR
set_files = sorted(glob.glob(os.path.join(DATA_DIR, "sub-*_task-ECOFF_run-1_preproc_raw.fif")))
CONDITION = "EC"
print("Found", len(set_files), "ECOFF files")
print("Example files:", set_files[:3])

# sanity check a single file
if len(set_files) > 0:
    print("First file exists:", os.path.exists(set_files[0]))

Found 28 ECOFF files
Example files: ['/content/drive/MyDrive/tiw_EO_EC_organized/preprocessed_organized/cleaned_organized/sub-001_task-ECOFF_run-1_preproc_raw.fif', '/content/drive/MyDrive/tiw_EO_EC_organized/preprocessed_organized/cleaned_organized/sub-002_task-ECOFF_run-1_preproc_raw.fif', '/content/drive/MyDrive/tiw_EO_EC_organized/preprocessed_organized/cleaned_organized/sub-004_task-ECOFF_run-1_preproc_raw.fif']
First file exists: True


In [ ]:
# helper functions
import numpy as np

def compute_acw(signal, sfreq):
    signal = np.asarray(signal, dtype=float)
    signal = signal - np.mean(signal)

    if np.allclose(signal, 0):
        return np.nan

    acf = np.correlate(signal, signal, mode='full')
    acf = acf[acf.size // 2:]

    if acf[0] == 0:
        return np.nan

    acf = acf / acf[0]

    below = np.where(acf < 0.5)[0]
    if len(below) == 0:
        return np.nan

    lag = below[0]
    acw = lag / sfreq
    return acw


def compute_acw_sliding(raw, window_sec=5, step_sec=1):
    """Compute ACW per EEG channel for each sliding window."""
    data = raw.get_data(picks='eeg')
    sfreq = raw.info['sfreq']

    window = int(window_sec * sfreq)
    step = int(step_sec * sfreq)
    n_samples = data.shape[1]

    acw_epochs = []

    for start in range(0, n_samples - window + 1, step):
        stop = start + window
        segment = data[:, start:stop]

        acw_channels = []
        for ch in segment:
            acw = compute_acw(ch, sfreq)
            acw_channels.append(acw)

        acw_epochs.append(acw_channels)

    return np.array(acw_epochs)


def summarize_acw_epochs(acw_epochs):
    mean_per_window = np.nanmean(acw_epochs, axis=1)   # ACW over time (avg across channels)
    mean_per_channel = np.nanmean(acw_epochs, axis=0)  # ACW over space (avg across windows)
    grand_mean = np.nanmean(acw_epochs)                # overall ACW (avg over everything)
    return mean_per_window, mean_per_channel, grand_mean


In [ ]:
# ============================================================
# D) SUBJECT LOOP: compute ROI timescales per subject
# ============================================================

all_subject_dfs = []  # collects each subject's df_tiw

for path_ec in set_files:

    print("\n====================================================")
    print("Processing:", os.path.basename(path_ec))
    print("====================================================")

    # ------------------------------------------------------------
    # 1) Load EEG
    # ------------------------------------------------------------
    raw = mne.io.read_raw_fif(path_ec, preload=True)
    print(raw)

    if "ECG" in raw.ch_names:
        raw.set_channel_types({"ECG": "ecg"})

    # ------------------------------------------------------------
    # 2) Resample to 128 Hz
    # ------------------------------------------------------------
    raw.resample(128)
    sfreq = raw.info['sfreq']
    print("New sampling rate =", sfreq)
    # ------------------------------------------------------------
    # 3) ACW per channel per window
    # ------------------------------------------------------------
    epochs_acw = mne.make_fixed_length_epochs(
        raw,
        duration=5.0,
        overlap=4.0,   # 5s window with 1s step
        preload=True
    )
    # channel-level loop
    ch_names = [epochs_acw.ch_names[i] for i in mne.pick_types(epochs_acw.info, eeg=True, exclude='bads')] #getting the names of good channels
    n_channels = len(ch_names)
    n_windows = len(epochs_acw)

    # shape: (n_windows, n_channels)
    acw_epochs = np.full((n_windows, n_channels), np.nan) #creating windows per channels metrix

    for wi, epoch in enumerate(epochs_acw.get_data(picks='eeg')):
        # epoch shape: (n_channels, n_times)
        for ci in range(n_channels):
            sig = epoch[ci, :] #selecting the channel
            acw_epochs[wi, ci] = compute_acw(sig, sfreq) #computing acw for channel in specific time window

    acw_mean_per_window, acw_mean_per_channel, acw_grand_mean = summarize_acw_epochs(acw_epochs)
    acw_std_per_channel = np.nanstd(acw_epochs, axis=0)
    ACW = {ch: acw_mean_per_channel[ci] for ci, ch in enumerate(ch_names)}
    ACW_std = {ch: acw_std_per_channel[ci]  for ci, ch in enumerate(ch_names)}
    # ------------------------------------------------------------
    # 4) Build per-subject ACW dataframes + save + append
    # ------------------------------------------------------------
    sub_id = os.path.basename(path_ec).split("_")[0]
    results = []
    for ch in ch_names:
        results.append({
            "Subject":           sub_id,
            "Condition":         CONDITION,
            "Channel":           ch,
            "ACW_mean_s":        ACW[ch],
            "ACW_std_s":         ACW_std[ch],
            "ACW_grand_mean_s":  acw_grand_mean
        })

    df_acw = pd.DataFrame(results)
    df_acw.to_csv(os.path.join(RESULTS_DIR, f"ACW_{sub_id}_{CONDITION}.csv"), index=False)
    all_subject_dfs.append(df_acw)

    # ---- File 2: ACW per window per channel (long format) ----
    rows = []
    for wi in range(n_windows):
        for ci, ch in enumerate(ch_names):
            rows.append({
                "Subject":   sub_id,
                "Condition": CONDITION,
                "Window":    wi,
                "Channel":   ch,
                "ACW_s":     acw_epochs[wi, ci]
            })

    df_acw_windows = pd.DataFrame(rows)
    df_acw_windows.to_csv(os.path.join(RESULTS_DIR, f"ACW_windows_{sub_id}_{CONDITION}.csv"), index=False)


Processing: sub-001_task-ECOFF_run-1_preproc_raw.fif
Opening raw data file /content/drive/MyDrive/tiw_EO_EC_organized/preprocessed_organized/cleaned_organized/sub-001_task-ECOFF_run-1_preproc_raw.fif...
    Range : 0 ... 960299 =      0.000 ...   192.060 secs
Ready.
Reading 0 ... 960299  =      0.000 ...   192.060 secs...
<Raw | sub-001_task-ECOFF_run-1_preproc_raw.fif, 64 x 960300 (192.1 s), ~469.0 MiB, data loaded>
New sampling rate = 128.0
Not setting metadata
188 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 188 events and 640 original time points ...
0 bad epochs dropped

Processing: sub-002_task-ECOFF_run-1_preproc_raw.fif
Opening raw data file /content/drive/MyDrive/tiw_EO_EC_organized/preprocessed_organized/cleaned_organized/sub-002_task-ECOFF_run-1_preproc_raw.fif...
    Range : 0 ... 1085599 =      0.000 ...   217.120 secs
Ready.
Reading 0 ... 1085599  =      0.000 ...   217.120 secs...
<Raw | sub-002_task

In [ ]:
# ============================================================
# E) GROUP-LEVEL: concatenate subjects, compute ROI means, and plot
# ============================================================

# ------------------------------------------------------------
# 1) Concatenate all subjects into a single dataframe
# ------------------------------------------------------------
df_all = pd.concat(all_subject_dfs, ignore_index=True)
df_all["Condition"] = CONDITION
df_all.to_csv(os.path.join(RESULTS_DIR, f"ACW_ALL_SUBJECTS_{CONDITION}.csv"), index=False)
print("\n================ GROUP-LEVEL SUMMARY ================")
print("Rows in df_all:", len(df_all))
print("Unique subjects in df_all:", df_all["Subject"].nunique())
print("First few subjects:", sorted(df_all["Subject"].unique())[:10], "...")
print("=====================================================\n")

print(df_all.head())


# ------------------------------------------------------------
# 2) channel-level means across subjects (ACW only)
# ------------------------------------------------------------
# df_all has ACW_mean_s and ACW_std_s per subject per channel
df_mean = (
    df_all.groupby(["Condition", "Channel"], as_index=False)
          .agg(
              ACW_mean_s       = ("ACW_mean_s",  "mean"),
              ACW_std_s        = ("ACW_std_s",   "mean"),   # mean of per-subject SDs
              ACW_grand_mean_s = ("ACW_grand_mean_s", "mean")
          )
)


================ GROUP-LEVEL SUMMARY ================
Rows in df_all: 1764
Unique subjects in df_all: 28
First few subjects: ['sub-001', 'sub-002', 'sub-004', 'sub-005', 'sub-006', 'sub-007', 'sub-008', 'sub-009', 'sub-010', 'sub-011'] ...

   Subject Condition Channel  ACW_mean_s  ACW_std_s  ACW_grand_mean_s
0  sub-001        EC     Fp1    0.027926   0.003863           0.02904
1  sub-001        EC     Fp2    0.023853   0.001930           0.02904
2  sub-001        EC      F3    0.031250   0.000000           0.02904
3  sub-001        EC      F4    0.031375   0.001705           0.02904
4  sub-001        EC      C3    0.031416   0.001127           0.02904


In [ ]:
# ============================================================
# F) CONTRIBUTION COUNTS
#    How many subjects contributed to each ROI’s mean?
#
# Two useful versions:
#   (1) ROI-level subject count overall (any row exists)
#   (2) ROI-level subject count per metric (non-NaN values)
# ============================================================

# (1) Overall: how many subjects are present for each channel (regardless of NaNs)
roi_n_subjects_overall = (
    df_all.groupby(["Channel"], as_index=False)["Subject"]
          .nunique()
          .rename(columns={"Subject": "N_Subjects_overall"})
)

# (2) Per metric: how many subjects have a valid value for each metric in each channel
roi_n_subjects_per_metric = {}

for m in ["ACW_mean_s", "ACW_std_s"]:
    roi_n_subjects_per_metric[m] = (
        df_all.dropna(subset=[m])
              .groupby(["Channel"], as_index=False)["Subject"]
              .nunique()
              .rename(columns={"Subject": f"N_Subjects_{m}"})
    )

# Merge counts into a single table
# Merge on "Channel" only (no "Cluster"):
df_counts = roi_n_subjects_overall.copy()
for m in ["ACW_mean_s", "ACW_std_s"]:
    df_counts = df_counts.merge(roi_n_subjects_per_metric[m], on=["Channel"], how="left")

# Join counts onto df_mean (so you can inspect means + N in one place)
df_mean_withN = df_mean.merge(df_counts, on=["Channel"], how="left")
df_mean_withN["Condition"] = CONDITION

# Save the “means + N” table
df_mean_withN.to_csv(os.path.join(RESULTS_DIR, f"ACW_CHANNEL_MEAN_WITH_N_{CONDITION}.csv"), index=False)
print(f"Saved channel mean + contribution counts to: {os.path.join(RESULTS_DIR, f'ACW_CHANNEL_MEAN_WITH_N_{CONDITION}.csv')}")
print(df_mean_withN[["Channel", "N_Subjects_overall",
                      "N_Subjects_ACW_mean_s", "N_Subjects_ACW_std_s"]].head())

Saved channel mean + contribution counts to: /content/drive/MyDrive/tiw_EO_EC_organized/acw_results/ACW_CHANNEL_MEAN_WITH_N_EC.csv
  Channel  N_Subjects_overall  N_Subjects_ACW_mean_s  N_Subjects_ACW_std_s
0     AF3                  28                     28                    28
1     AF4                  28                     28                    28
2     AF7                  28                     28                    28
3     AF8                  28                     28                    28
4      C1                  28                     28                    28
